In [2]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [4]:
X_train = np.load('/content/X_train_scaled.npy')
X_val = np.load('/content/X_val_scaled.npy')
X_test = np.load('/content/X_test_scaled.npy')
y_train = np.load('/content/y_train.npy')
y_val = np.load('/content/y_val.npy')

In [41]:
class DefaultRiskPredictor(nn.Module):
  def __init__(self, input_dim=135, hidden_dim=256, dropout=0.3):
    super().__init__()
    self.seq = nn.Sequential(
        nn.Linear(input_dim, hidden_dim, bias=False),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(hidden_dim, 128, bias=False),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(128, 1)
    )

  def forward(self, x):
    x = self.seq(x)
    return x

In [42]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32, device=device)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32, device=device).view(-1, 1)

X_val_tensor = torch.tensor(X_val, dtype=torch.float32, device=device)
y_val_tensor = torch.tensor(y_val, dtype=torch.float32, device=device).view(-1, 1)

X_test_tensor = torch.tensor(X_test, dtype=torch.float32, device=device)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)

train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [43]:
model = DefaultRiskPredictor().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.005)
scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
scale = (y_train == 0).sum() / (y_train == 1).sum()
pos_weight = torch.tensor(scale).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

def train_epoch(model, dataloader, optimizer, epoch, device):
  model.train()

  train_loss = 0
  for batch in dataloader:
    optimizer.zero_grad()
    inputs = batch[0].to(device)
    labels = batch[1].to(device)
    outputs = model(inputs)
    loss = criterion(outputs, labels)
    train_loss += loss.item()
    loss.backward()
    optimizer.step()

  avg_loss = train_loss / len(dataloader)
  print(f'Epoch {epoch+1}: Train Loss = {avg_loss:.3f}')
  return avg_loss

def evaluate(model, dataloader, epoch, device):
  model.eval()
  val_loss = 0
  all_preds, all_labels = [], []
  with torch.no_grad():
    for batch in dataloader:
      inputs = batch[0].to(device)
      labels = batch[1].to(device)
      outputs = model(inputs)
      loss = criterion(outputs, labels)
      val_loss += loss.item()
      all_preds.append(outputs)
      all_labels.append(labels)

  avg_loss = val_loss / len(dataloader)
  all_logits = torch.sigmoid(torch.cat(all_preds, dim=0)).cpu().numpy().squeeze()
  pred_classes = (all_logits > 0.5).astype(int)
  all_labels = torch.cat(all_labels, dim=0).cpu().numpy().squeeze()
  val_f1 = f1_score(all_labels, pred_classes, average='macro')
  print(f'Epoch {epoch+1}: Validation Loss = {avg_loss:.3f}, F1 = {val_f1:.3f}')
  return avg_loss, val_f1

In [44]:
test_ids = np.load('/content/test_ids.npy')
test_ids

array([100001, 100005, 100013, ..., 456223, 456224, 456250], dtype=int32)

In [45]:
epochs = 100
best_f1 = 0
early_stop_counter = 0
patience = 10
for epoch in range(epochs):
  train_loss = train_epoch(model, train_dataloader, optimizer, epoch, device)
  val_loss, val_f1 = evaluate(model, val_dataloader, epoch, device)
  if val_f1 > best_f1:
    best_f1 = val_f1
    torch.save(model.state_dict(), 'best_model.pt')
    early_stop_counter = 0
  else:
    early_stop_counter += 1
    if early_stop_counter == patience:
      print(f'Early stopping at epoch {epoch+1}')
      break
  scheduler.step(val_f1)


model.load_state_dict(torch.load('best_model.pt'))
model.eval()

test_outputs = model(X_test_tensor)
test_probs = torch.sigmoid(test_outputs).detach().cpu().numpy().squeeze()

res_df = pd.DataFrame({
    'SK_ID_CURR': test_ids,
    'TARGET': test_probs
})

res_df.to_csv('submission.csv', index=False)

Epoch 1: Train Loss = 1.122
Epoch 1: Validation Loss = 1.106, F1 = 0.511
Epoch 2: Train Loss = 1.101
Epoch 2: Validation Loss = 1.081, F1 = 0.551
Epoch 3: Train Loss = 1.094
Epoch 3: Validation Loss = 1.074, F1 = 0.521
Epoch 4: Train Loss = 1.090
Epoch 4: Validation Loss = 1.097, F1 = 0.575
Epoch 5: Train Loss = 1.084
Epoch 5: Validation Loss = 1.066, F1 = 0.546
Epoch 6: Train Loss = 1.082
Epoch 6: Validation Loss = 1.104, F1 = 0.595
Epoch 7: Train Loss = 1.079
Epoch 7: Validation Loss = 1.064, F1 = 0.549
Epoch 8: Train Loss = 1.075
Epoch 8: Validation Loss = 1.063, F1 = 0.542
Epoch 9: Train Loss = 1.072
Epoch 9: Validation Loss = 1.061, F1 = 0.546
Epoch 10: Train Loss = 1.072
Epoch 10: Validation Loss = 1.064, F1 = 0.554
Epoch 11: Train Loss = 1.068
Epoch 11: Validation Loss = 1.064, F1 = 0.542
Epoch 12: Train Loss = 1.066
Epoch 12: Validation Loss = 1.069, F1 = 0.562
Epoch 13: Train Loss = 1.057
Epoch 13: Validation Loss = 1.064, F1 = 0.560
Epoch 14: Train Loss = 1.053
Epoch 14: Vali